# Codificação de Sinais Multimedia

## Trabalho Laboratorial 3

- 52146 - Margarida Andrade
- 52345 - Jorge Gonçalves
- 52708 - Helena Nina

In [2]:
import numpy as np
import matplotlib.pyplot as plt

## Exercício 1


### Exercício 1

Nesta alínea é construída uma tabela de Huffman a partir de símbolos e respetivos pesos (probabilidades ou ocorrências). A função devolve uma tabela com o código binário associado a cada símbolo e pode ser aplicada diretamente a um histograma de frequência.

In [3]:
from collections import Counter
from heapq import heappop, heappush
from itertools import count
from pathlib import Path


def gen_huff_table(symbols, probabilities=None):
    """Gera um dicionario simbolo -> codigo de Huffman.
        symbols: lista de simbolos ou dicionario {simbolo: probabilidade}
        probabilities: lista de probabilidades (opcional se symbols é dict)
    """
    if isinstance(symbols, dict):
        freq = symbols
    else:
        if probabilities is None:
            raise ValueError("probabilities deve ser fornecido quando symbols nao e um dicionario")
        freq = dict(zip(symbols, probabilities))
    
    heap = []
    order = count()
    for symbol, weight in freq.items():
        heappush(heap, (float(weight), next(order), symbol))
    
    if len(heap) == 1:
        return {heap[0][2]: "0"}
    
    while len(heap) > 1:
        w1, _, left = heappop(heap)
        w2, _, right = heappop(heap)
        heappush(heap, (w1 + w2, next(order), (left, right)))
    
    codes = {}
    def walk(node, prefix=""):
        if isinstance(node, tuple):
            walk(node[0], prefix + "0")
            walk(node[1], prefix + "1")
        else:
            codes[node] = prefix or "0"
    
    walk(heap[0][2])
    return codes


texto = Path("dados-CSM-TP3-Huffman/DecUniversalDH.txt").read_text(encoding="utf-8")
freq = Counter(texto)
total = sum(freq.values())
probs = {symbol: count / total for symbol, count in freq.items()}
codes = gen_huff_table(probs)

print(f"{'simbolo':>10} {'prob':>8} {'codigo'}")
for symbol, prob in sorted(probs.items(), key=lambda x: (-x[1], str(x[0]))):
    print(f"{repr(symbol):>10} {prob:8.4f} {codes[symbol]}")

   simbolo     prob codigo
       ' '   0.1632 110
       'e'   0.0991 001
       'a'   0.0859 000
       'o'   0.0841 1110
       'i'   0.0635 1010
       's'   0.0632 1000
       'd'   0.0536 0110
       'r'   0.0531 0101
       't'   0.0416 11110
       'n'   0.0389 10110
       'm'   0.0296 10010
       'u'   0.0280 01110
       'c'   0.0249 01000
       'l'   0.0222 111111
       'p'   0.0203 101111
      '\n'   0.0139 011110
       'g'   0.0101 1111100
       ','   0.0100 1011101
       'v'   0.0095 1011100
       '.'   0.0078 1001100
       'ç'   0.0078 0111111
       'ã'   0.0075 0111110
       'f'   0.0062 0100101
       'b'   0.0060 0100100
       'q'   0.0049 10011111
       'h'   0.0040 10011011
       'í'   0.0036 01001111
       'A'   0.0035 01001101
       'é'   0.0027 111110111
       'T'   0.0026 111110101
       'º'   0.0026 111110100
       'à'   0.0022 100111001
       'õ'   0.0022 100111010
       'á'   0.0020 100111000
       'z'   0.0019 100110100
       'j'   0.

## Exercício 2


In [4]:
def encode_huff(message, codes):
    """Codifica uma mensagem usando a tabela de Huffman.
    
    Args:
        message: string com a mensagem a codificar
        codes: dicionario {simbolo: codigo_binario}
    
    Returns:
        string com bits codificados (sequencia de 0s e 1s)
    """
    encoded = ""
    for symbol in message:
        if symbol not in codes:
            raise ValueError(f"Símbolo '{symbol}' não encontrado na tabela de Huffman")
        encoded += codes[symbol]
    return encoded


# Teste da função com a mensagem do arquivo
mensagem_teste = texto[:100]  # Pega os primeiros 100 caracteres
encoded = encode_huff(mensagem_teste, codes)

print(f"Mensagem original: {repr(mensagem_teste)}")
print(f"Tamanho original: {len(mensagem_teste) * 8} bits (8 bits por símbolo)")
print(f"Tamanho codificado: {len(encoded)} bits")
print(f"Taxa de compressão: {len(encoded) / (len(mensagem_teste) * 8) * 100:.2f}%")
print(f"\nPrimeiros 100 bits codificados: {encoded[:100]}")

Mensagem original: '\nDeclaração Universal dos Direitos Humanos\nPreâmbulo\n\nConsiderando que o reconhecimento da dignidade'
Tamanho original: 800 bits (8 bits por símbolo)
Tamanho codificado: 478 bits
Taxa de compressão: 59.75%

Primeiros 100 bits codificados: 0111100100111011001010001111110000101000011111101111101110110100111101111011010101011100001010110000


## Exercício 3

In [8]:
def decode_huff(encoded_message, codes):
    """Descodifica uma mensagem codificada com Huffman.
    
    Args:
        encoded_message: string com bits codificados (sequencia de 0s e 1s)
        codes: dicionario {simbolo: codigo_binario}
    
    Returns:
        string com a mensagem descodificada (símbolos originais)
    """
    # Inverte a tabela: código -> símbolo
    reverse_codes = {code: symbol for symbol, code in codes.items()}
    
    decoded = ""
    current_code = ""
    
    for bit in encoded_message:
        current_code += bit
        if current_code in reverse_codes:
            decoded += reverse_codes[current_code]
            current_code = ""
    
    if current_code:  # Se sobrarem bits no final
        raise ValueError(f"Código inválido no final da mensagem: '{current_code}'")
    
    return decoded


# Teste: verificar que decode_huff(encode_huff(msg)) == msg
mensagem_original = texto[:100]
encoded = encode_huff(mensagem_original, codes)
decoded = decode_huff(encoded, codes)

print(f"Mensagem original: {repr(mensagem_original)}")
print(f"Mensagem descodificada: {repr(decoded)}")
print(f"\nMensagens são iguais: {mensagem_original == decoded}")
print(f"Comprimento original: {len(mensagem_original)}")
print(f"Comprimento descodificado: {len(decoded)}")

Mensagem original: '\nDeclaração Universal dos Direitos Humanos\nPreâmbulo\n\nConsiderando que o reconhecimento da dignidade'
Mensagem descodificada: '\nDeclaração Universal dos Direitos Humanos\nPreâmbulo\n\nConsiderando que o reconhecimento da dignidade'

Mensagens são iguais: True
Comprimento original: 100
Comprimento descodificado: 100


## Exercício 4

In [14]:
# Exercício 4: Codificar a tabela de Huffman

def encode_table(codes):
    """Codifica a tabela de Huffman numa sequência de bits.
    
    Formato da sequência binária:
    - 16 bits: número de símbolos
    - Para cada símbolo:
        * 8 bits: valor ASCII do símbolo
        * 8 bits: comprimento do código Huffman
        * N bits: o código Huffman em si
    
    Args:
        codes: dicionário {símbolo: código_binário}
    
    Returns:
        string com bits (0s e 1s) da tabela codificada
    """
    encoded_table = ""
    
    # Adiciona número de símbolos (16 bits)
    encoded_table += format(len(codes), '016b')
    
    # Codifica cada símbolo e seu código
    for symbol, code in codes.items():
        encoded_table += format(ord(symbol), '08b')  # Símbolo em 8 bits
        encoded_table += format(len(code), '08b')     # Comprimento do código em 8 bits
        encoded_table += code                        # Código Huffman
    
    return encoded_table


# Codificar a tabela
encoded_table = encode_table(codes)
print(f"Tabela Huffman codificada: {len(encoded_table)} bits")
print(f"Primeiros 64 bits: {encoded_table[:64]}")

Tabela Huffman codificada: 1586 bits
Primeiros 64 bits: 0000000001000000011000010000001100001100101000000110010110001100


In [15]:
# Acrescentar a sequência binária da tabela à mensagem codificada (do Ex. 2)
# Resultado final: [tabela codificada] + [mensagem codificada]
full_encoded = encoded_table + encoded

print(f"\nTamanho da tabela: {len(encoded_table)} bits")
print(f"Tamanho da mensagem (Ex. 2): {len(encoded)} bits")
print(f"Tamanho total: {len(full_encoded)} bits")
print(f"\nComposição: tabela {len(encoded_table) / len(full_encoded) * 100:.1f}% + mensagem {len(encoded) / len(full_encoded) * 100:.1f}%")
print(f"\nResultado final (primeiros 100 bits):")
print(full_encoded[:100])


Tamanho da tabela: 1586 bits
Tamanho da mensagem (Ex. 2): 478 bits
Tamanho total: 2064 bits

Composição: tabela 76.8% + mensagem 23.2%

Resultado final (primeiros 100 bits):
0000000001000000011000010000001100001100101000000110010110001100000101010000110001000000111010010001


In [16]:
# [Opcional] Para descodificar a tabela e reconstruir o dicionário de Huffman,
# seria necessário inverter o processo: ler 16 bits iniciais (nº símbolos),
# depois para cada símbolo ler 8+8+N bits (símbolo, comprimento, código)

## Exercício 5

## Exercício 6

## Exercício 7
### Alínea a

### Alínea b

### Alínea c

### Alínea d

### Alínea e

### Alínea f

### Alínea g